# Подготовка данных для обучения моделей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://scikit-learn.org/stable/modules/compose.html#pipeline-chaining-estimators
* https://pytorch.org/docs/stable/data.html
* https://pytorch.org/tutorials/beginner/data_loading_tutorial.html
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann


## Задачи для совместного разбора

1. Создайте синтетический датасет для задачи регрессии и представьте его в виде `torch.utils.data.Dataset`

In [ ]:
import torch
import pandas as pd
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset

In [ ]:
n_samples = 1000
n_features = 5
noise_std = 0.1

X = torch.randn(n_samples, n_features)
weights = torch.randn(n_features)
bias = torch.randn(1)
y = torch.matmul(X, weights) + bias
y += torch.normal(mean=0, std=noise_std, size=(n_samples,))

dataset = TensorDataset(X, y)
X.shape, y.shape

(torch.Size([1000, 5]), torch.Size([1000]))

In [ ]:
for i in range(3):
    x, y_val = dataset[i]
    print(f"Пример {i}: X={x.tolist()}, y={y_val.item():.4f}")

Пример 0: X=[0.03284618258476257, 0.4596549868583679, 1.437056541442871, 0.5399503707885742, 0.484992116689682], y=-1.6834
Пример 1: X=[-1.5641835927963257, -0.5771201848983765, 0.37319761514663696, 0.14092819392681122, 1.393959641456604], y=-1.3275
Пример 2: X=[0.33529797196388245, 1.117712378501892, 0.2409846931695938, 0.5535575151443481, 0.19057409465312958], y=-0.1517


## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Считайте файл `bank-full.csv` ([источник](https://www.kaggle.com/datasets/hariharanpavan/bank-marketing-dataset-analysis-classification)) в виде `pd.DataFrame`.

Опишите класс `BankDatasetBase`. Решение должно удовлетворять следующим критериям:

* класс наследуется от `torch.utils.data.Dataset`;
* при создании объекта в конструктор передается набор данных в виде `pd.DataFrame`;
* объекты класса имеют поля `X` и `y` с признаками и метками соответственно;
* класс реализует интерфейс последовательностей (`__getitem__` + `__len__`);
* `obj[i]` возвращает кортеж, содержащий `i`-ую строку из `obj.X` (серию) и `i`-ую строку из `obj.y` (строку).
    
Создайте объект класса `BankDatasetBase` и продемонстрируйте работоспособность.

- [ ] Проверено на семинаре

In [ ]:
class BankDatasetBase(Dataset):
    def __init__(self, data: pd.DataFrame) -> None:
        self.X = data.drop(columns = ['y'])
        self.y = data['y']
        self.n_samples = len(data)

    def __getitem__(self, idx: int) -> tuple:
        return self.X.iloc[idx], self.y.iloc[idx]

    def __len__(self) -> int:
        return self.n_samples

In [ ]:
data = pd.read_csv('bank-full.csv')
dataset = BankDatasetBase(data)

print(f"Размер датасета: {len(dataset)}")
print(f"Признаки (первые 5 строк):\n{dataset.X.head()}")
print(f"Метки (первые 5):\n{dataset.y.head()}")

Размер датасета: 45211
Признаки (первые 5 строк):
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome  
0  unknown    5   may       261         1     -1         0  unknown  
1  unknown    5   may       151         1     -1         0  unknown  
2  unknown    5   may        76         1     -1         0  unknown  
3  unknown    5   may        92         1     -1         0  unknown  
4  unknown    5   may       198         1     -1         0  unknown  
Метки (первые 5):
0    no
1    no
2    no
3    no
4    no
Name: y, dtype: object


<p class="task" id="2"></p>

2\. Опишите класс `BankDataset`. Решение должно удовлетворять всем критериям из предыдущего задания, а также:
* при создании объекта в конструктор может быть передан необязательные аргументы `transform` и `target_transform`;
* если аргумент `transform` был передан, то при получении `i`-го элемента, нужно вызвать `transform(x)` и вернуть полученный результат.
* если аргумент `target_transform` был передан, то при получении `i`-го элемента, нужно вызвать `target_transform(y)` и вернуть полученный результат.

Создайте объект класса `BankDataset` и продемонстрируйте работоспособность (без передачи `target_transform` и `transform`).

- [ ] Проверено на семинаре

In [ ]:
class BankDataset(Dataset):
    def __init__(
            self,
            data: pd.DataFrame,
            transform: Callable | None = None,
            target_transform: Callable | None = None,
            label_column: str = 'y'
    ) -> None:

        self.X = data.drop(columns=[label_column])
        self.y = data[label_column]
        self.n_samples = len(data)
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, idx: int) -> tuple:
        x = self.X.iloc[idx]
        y = self.y.iloc[idx]

        if self.transform is not None:
            x = self.transform(x)  # transform применяется только к x
        if self.target_transform is not None:
            y = self.target_transform(y)  # target_transform применяется только к y

        return x, y

    def __len__(self) -> int:
        return self.n_samples

In [ ]:
data = pd.read_csv('bank-full.csv')
print(data.head())

dataset = BankDataset(data, label_column='y')

print(f"\nРазмер датасета: {len(dataset)}")
print(f"Признаки (первые 5 строк):\n{dataset.X.head()}")

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  

Размер датасета: 45211
Признаки (первые 5 строк):
   age           job  marital  education default  balance ho

In [ ]:
def example_transform(x: pd.Series) -> dict:
    return {'age': x['age'], 'balance': x['balance']}
def example_target_transform(y: str) -> int:
    return 1 if y == 'yes' else 0
dataset_with_transforms = BankDataset(data, transform=example_transform, target_transform=example_target_transform)

# Проверяем
x, y = dataset_with_transforms[0]
print(f"Признаки (после transform): {x}")
print(f"Метка (после target_transform): {y}")

Признаки (после transform): {'age': np.int64(58), 'balance': np.int64(2143)}
Метка (после target_transform): 0


<p class="task" id="3"></p>

3\. Опишите класс `OrdinalEncoderTransform`. Решение должно удовлетворять следующим критериям:

* при создании объекта в конструктор передаются названия нечисловых столбцов в датасете
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (признаки) и возвращает набор признаков, в котором нечисловые характеристики закодированы целыми числами;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объект класса `OrdinalEncoderTransform`.

- [ ] Проверено на семинаре

In [ ]:
class Transform:
    pass

In [ ]:
import pandas as pd

class OrdinalEncoderTransform(Transform):
    def __init__(self, category_columns: list[str]) -> None:
        self.category_columns = category_columns
        self.category_to_index = {col: {} for col in category_columns}
        self.next_index = {col: 0 for col in category_columns}
    def __call__(self, x: pd.Series) -> pd.Series:
        x_transformed = x.copy()
        for col in self.category_columns:
            value = x[col]
            if value not in self.category_to_index[col]:
                self.category_to_index[col][value] = self.next_index[col]
                self.next_index[col] += 1
            x_transformed[col] = self.category_to_index[col][value]
        return x_transformed

In [ ]:
data = pd.read_csv('bank-full.csv')
category_columns = ['job', 'marital', 'education', 'default', 'housing',
                           'loan', 'contact', 'month', 'poutcome']

transform = OrdinalEncoderTransform(category_columns=category_columns)

dataset = BankDataset(data, transform=transform, label_column='y')

print(f"Размер датасета: {len(dataset)}")
print(f"Признаки (первые 5 строк):\n{dataset.X.head()}")

for i in range(3):
    x, y = dataset[i]
    print(f"Пример {i}:")
    print(f"  Признаки (pd.Series): {x.to_dict()}")
    print(f"  Метка: {y}")


print("\nСловарь категорий для 'job':")
print(transform.category_to_index['job'])

Размер датасета: 45211
Признаки (первые 5 строк):
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome  
0  unknown    5   may       261         1     -1         0  unknown  
1  unknown    5   may       151         1     -1         0  unknown  
2  unknown    5   may        76         1     -1         0  unknown  
3  unknown    5   may        92         1     -1         0  unknown  
4  unknown    5   may       198         1     -1         0  unknown  
Пример 0:
  Признаки (pd.Series): {'age': 58, 'job': 0, 'marital': 0, 'education': 0,

<p class="task" id="4"></p>

4\. Опишите класс `LabelEncoderTransform`. Решение должно удовлетворять следующим критериям:

* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` имеет один параметр (строку) и возвращает целое число, соответствующее этой строке;
* состояние объекта (индексы для кодирования) обновляется в момент очередного вызова `__call__` (т.е. все данные сразу никогда не передаются никакому методу объекта).
    
Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании объекта в качестве аргумента `target_transform` объект класса `LabelEncoderTransform`.

- [ ] Проверено на семинаре

In [ ]:
class LabelEncoderTransform(Transform):
    def __init__(self) -> None:
        self.label_to_index = {}
        self.next_index = 0

    def __call__(self, x: str) -> int:
        if x not in self.label_to_index:
            self.label_to_index[x] = self.next_index
            self.next_index += 1
        return self.label_to_index[x]

In [ ]:
data = pd.read_csv('bank-full.csv')
target_transform = LabelEncoderTransform()

dataset = BankDataset(data, target_transform=target_transform, label_column='y')


print(f"Размер датасета: {len(dataset)}")
print(f"Признаки (первые 5 строк):\n{dataset.X.head()}")
print(f"Метки (первые 5):\n{dataset.y.head()}")
print("\nПример данных (первые 3):")
for i in range(3):
            x, y = dataset[i]
            print(f"Пример {i}:")
            print(f"  Признаки (pd.Series): {x.to_dict()}")
            print(f"  Метка (после target_transform): {y}")
print("\nСловарь меток:")
print(target_transform.label_to_index)

Размер датасета: 45211
Признаки (первые 5 строк):
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome  
0  unknown    5   may       261         1     -1         0  unknown  
1  unknown    5   may       151         1     -1         0  unknown  
2  unknown    5   may        76         1     -1         0  unknown  
3  unknown    5   may        92         1     -1         0  unknown  
4  unknown    5   may       198         1     -1         0  unknown  
Метки (первые 5):
0    no
1    no
2    no
3    no
4    no
Name: y, dtype: object

При

<p class="task" id="5"></p>

5\. Опишите класс `ToTensor`.  Решение должно удовлетворять следующим критериям:
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает на вход серию или фрейм и возвращает тензор.

Опишите класс `Compose`.  Решение должно удовлетворять следующим критериям:
* при создании объекта в конструктор передается список объектов `transforms`, каждый из которых имеет метод `__call__(x, y)`;
* класс реализует интерфейс `Callable` (`__call__`); метод `__call__` принимает имеет параметра (признаки и класс в числовом виде) и и возвращает кортеж, полученный путем последовательного вызова объектов из `transforms`.

Продемонстрируйте работоспособность, создав объект `BankDataset` и передав при создании преобразования `Compose` список из объектов LabelEncoderTransform и ToTensor.

- [ ] Проверено на семинаре

In [ ]:
import torch as th
from typing import Any

class ToTensor(Transform):
    def __call__(self, x: Any) -> th.Tensor:
        if isinstance(x, pd.Series):
            numeric_values = []
            for value in x.values:
                if isinstance(value, (int, float, np.number)):
                    numeric_values.append(float(value))
                else:
                    try:
                        numeric_values.append(float(value))
                    except (ValueError, TypeError):
                        numeric_values.append(0.0)
            return th.tensor(numeric_values, dtype=th.float32)
        elif isinstance(x, int):
            return th.tensor(x, dtype=th.long)
        else:
            return th.tensor(float(x), dtype=th.float32)

class Compose(Transform):
    def __init__(self, transforms: list[Transform]) -> None:
        self.transforms = transforms

    def __call__(self, x: Any) -> Any:
        for transform in self.transforms:
            x = transform(x)
        return x

In [ ]:
data = pd.read_csv('bank-full.csv')
category_columns = ['job', 'marital', 'education', 'default', 'housing',
                           'loan', 'contact', 'month', 'poutcome']

ordinal_encoder = OrdinalEncoderTransform(category_columns=category_columns)
to_tensor = ToTensor()
label_encoder = LabelEncoderTransform()

feature_compose = Compose(transforms=[ordinal_encoder, to_tensor])

target_compose = Compose(transforms=[label_encoder, to_tensor])

dataset = BankDataset(
    data,
    transform=feature_compose,
    target_transform=target_compose,
    label_column='y'
)

print("\nПример данных (первые 3):")
for i in range(3):
    x, y = dataset[i]
    print(f"Пример {i}:")
    print(f"  Размер признаков: {x.shape}")
    print(f"  Метка: {y}")

print("\nСловарь меток:")
print(label_encoder.label_to_index)


Пример данных (первые 3):
Пример 0:
  Размер признаков: torch.Size([16])
  Метка: 0
Пример 1:
  Размер признаков: torch.Size([16])
  Метка: 0
Пример 2:
  Размер признаков: torch.Size([16])
  Метка: 0

Словарь меток:
{'no': 0}


<p class="task" id="6"></p>

6\. Разделите датасет из предыдущего задания на обучающую и тестовую выборку в соотношении 75% на 25%. Создайте объект `DataLoader` для получения пакетов размера 64, полученных из перемешанного обучающего датасета. Кастомизируйте `DataLoader` таким образом, чтобы пакет признаков был представлен в виде трехмерного тензора размера 64x2x8 (разделите 16 признаков на два тензора по 8). Получите один пакет и выведите на экран размерность тензоров пакета.

- [ ] Проверено на семинаре

In [ ]:
import torch as th
from torch.utils.data import DataLoader, random_split
import numpy as np

# 1. Разделяем датасет на обучающую и тестовую выборку
dataset_size = len(dataset)
train_size = int(0.75 * dataset_size)
test_size = dataset_size - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f"Общий размер датасета: {dataset_size}")
print(f"Обучающая выборка: {len(train_dataset)}")
print(f"Тестовая выборка: {len(test_dataset)}")

# 2. Создаем кастомную функцию collate для изменения формы тензоров
def custom_collate_fn(batch):
    """
    Функция для обработки пакета данных
    Преобразует тензоры признаков в форму 64x2x8
    """
    # batch - список кортежей (x, y)
    features = th.stack([item[0] for item in batch])  # [64, 16]
    labels = th.stack([item[1] for item in batch])    # [64]

    # Изменяем форму тензора признаков: [64, 16] -> [64, 2, 8]
    # Разделяем 16 признаков на 2 группы по 8 признаков
    batch_size = features.shape[0]
    features_reshaped = features.view(batch_size, 2, 8)

    return features_reshaped, labels

# 3. Создаем DataLoader для обучающей выборки
train_dataloader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=custom_collate_fn
)

# 4. Создаем обычный DataLoader для тестовой выборки (без изменения формы)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

# 5. Получаем один пакет из обучающего DataLoader
train_features, train_labels = next(iter(train_dataloader))

print("\n" + "="*50)
print("РЕЗУЛЬТАТ:")
print("="*50)
print(f"Размерность тензора признаков: {train_features.shape}")  # Должно быть [64, 2, 8]
print(f"Размерность тензора меток: {train_labels.shape}")        # Должно быть [64]
print(f"Тип тензора признаков: {train_features.dtype}")
print(f"Тип тензора меток: {train_labels.dtype}")

# 6. Дополнительная информация о пакете
print(f"\nДополнительная информация:")
print(f"Количество пакетов в обучающем DataLoader: {len(train_dataloader)}")
print(f"Количество пакетов в тестовом DataLoader: {len(test_dataloader)}")

# 7. Проверяем, что форма соответствует требованиям
assert train_features.shape == (64, 2, 8), f"Ожидалась форма (64, 2, 8), получено {train_features.shape}"
assert train_labels.shape == (64,), f"Ожидалась форма (64,), получено {train_labels.shape}"
print("\n✓ Форма тензоров соответствует требованиям!")

# 8. Пример содержимого
print(f"\nПример первых 3 строк признаков (оригинальная форма):")
print(train_features[:3])
print(f"\nСоответствующие метки:")
print(train_labels[:3])

Общий размер датасета: 45211
Обучающая выборка: 33908
Тестовая выборка: 11303

РЕЗУЛЬТАТ:
Размерность тензора признаков: torch.Size([64, 2, 8])
Размерность тензора меток: torch.Size([64])
Тип тензора признаков: torch.float32
Тип тензора меток: torch.int64

Дополнительная информация:
Количество пакетов в обучающем DataLoader: 530
Количество пакетов в тестовом DataLoader: 177

✓ Форма тензоров соответствует требованиям!

Пример первых 3 строк признаков (оригинальная форма):
tensor([[[ 5.2000e+01,  3.0000e+00,  1.0000e+00,  2.0000e+00,  0.0000e+00,
           2.2270e+03,  1.0000e+00,  0.0000e+00],
         [ 1.0000e+00,  2.4000e+01,  1.0000e+00,  2.4200e+02,  2.0000e+00,
          -1.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 3.1000e+01,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           1.5500e+02,  0.0000e+00,  0.0000e+00],
         [ 1.0000e+00,  8.0000e+00,  2.0000e+00,  2.1400e+02,  1.0000e+00,
           9.9000e+01,  1.0000e+00,  1.0000e+00]],

        [[ 3.8000e+